In [1]:
!unzip /content/augmented_german_clear_non-reply.zip

Archive:  /content/augmented_german_clear_non-reply.zip
   creating: content/augmented_german_clear_non-reply/
   creating: content/augmented_german_clear_non-reply/train/
  inflating: content/augmented_german_clear_non-reply/train/data-00000-of-00001.arrow  
  inflating: content/augmented_german_clear_non-reply/train/dataset_info.json  
  inflating: content/augmented_german_clear_non-reply/train/state.json  
   creating: content/augmented_german_clear_non-reply/test/
  inflating: content/augmented_german_clear_non-reply/test/data-00000-of-00001.arrow  
  inflating: content/augmented_german_clear_non-reply/test/dataset_info.json  
  inflating: content/augmented_german_clear_non-reply/test/state.json  
 extracting: content/augmented_german_clear_non-reply/dataset_dict.json  


In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a variable for your desired save path
drive_save_path = "/content/drive/My Drive/ModernBERT_QEvasion_Best_Model_0.70"

Mounted at /content/drive


In [3]:
from datasets import load_dataset, load_from_disk
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix # Added import

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the dataset
dataset = load_from_disk("/content/content/augmented_german_clear_non-reply")
print("Loaded back-translation augmented dataset from disk.")

# Prepare labels
labels = dataset['train'].unique('clarity_label')
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def add_labels(example):
    example['labels'] = label2id[example['clarity_label']]
    return example

dataset = dataset.map(add_labels)
dataset = dataset.remove_columns([
    col for col in dataset['train'].column_names if col not in ['question', 'interview_answer', 'labels']
])

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")


# Calculate class weights for imbalanced data
def get_class_weights(dataset, num_labels):
    label_counts = Counter(dataset["train"]["labels"])
    total_samples = len(dataset["train"])

    # Calculate class weights (inverse frequency)
    class_weights = []
    for i in range(num_labels):
        count = label_counts.get(i, 1)  # avoid division by zero
        weight = total_samples / (num_labels * count)
        class_weights.append(weight)

    # Move the tensor to the active device (GPU)
    return torch.tensor(
        class_weights, dtype=torch.float32, device=device
    )


# Get class weights
class_weights = get_class_weights(dataset, num_labels)
print(f"Class weights: {class_weights}")
print(f"Class weights device: {class_weights.device}")  # Verify it's on CUDA

# Focal Loss implementation :cite[1]:cite[8]
class FocalLoss(nn.Module):
    """
    Multi-class Focal loss implementation
    Focal loss helps address class imbalance by focusing on hard examples

    Args:
        gamma (float): Focusing parameter, higher values put more focus on hard examples
        weight (Tensor): Class weights tensor for handling imbalanced data
        ignore_index (int): Index to ignore in loss calculation
    """
    def __init__(self, gamma=1.0, weight=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.ignore_index = ignore_index

    def forward(self, input, target):
        # Calculate cross entropy loss
        ce_loss = F.cross_entropy(input, target, reduction='none', weight=self.weight, ignore_index=self.ignore_index)

        # Get probabilities
        pt = torch.exp(-ce_loss)

        # Compute focal loss
        focal_loss = (1 - pt) ** self.gamma * ce_loss

        return focal_loss.mean()


# Updated Custom Trainer with corrected compute_loss signature
class CustomTrainer(Trainer):
    """
    Custom trainer that uses Focal Loss with class weights
    This subclass overrides the compute_loss method to use our custom loss function
    """

    def __init__(self, *args, class_weights=None, focal_gamma=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.focal_loss = FocalLoss(gamma=focal_gamma, weight=class_weights)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        # Extract labels and run model forward pass
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Compute focal loss with class weights
        loss = self.focal_loss(logits, labels)

        # Handle return_outputs as required by the Trainer
        return (loss, outputs) if return_outputs else loss

# Tokenization and model setup
model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples['question'],
        examples['interview_answer'],
        truncation=True,
        padding="max_length",
        max_length=1680
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1_weighted = f1_score(labels, predictions, average='weighted', zero_division=0)

    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted
    }

# --- MODIFICATION 1: Updated TrainingArguments ---
# We now evaluate, save, and load the best model based on 'f1_macro'
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",          # <--- MODIFIED (was "no")
    save_strategy="epoch",
    load_best_model_at_end=True,    # <--- MODIFIED (was False)
    metric_for_best_model="f1_macro", # <--- NEW
    greater_is_better=True,         # <--- NEW
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# --- MODIFICATION 2: Updated CustomTrainer instantiation ---
# We pass the test set to eval_dataset
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],  # <--- NEW
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,  # Pass the calculated class weights
    focal_gamma=1.0,  # You can adjust this parameter
)

print(f"Using class weights: {class_weights}")
print("Starting training with Focal Loss (evaluating on test set after each epoch)...")
trainer.train()

print("Training completed!")

# --- MODIFICATION 3: Updated Final Evaluation ---
# This will now evaluate the *best* model saved during training
# (because of load_best_model_at_end=True)
test_results = trainer.evaluate() # <--- MODIFIED (no arg needed)
print("\n" + "="*60)
print(f"FINAL TEST RESULTS (from best epoch: {trainer.state.best_model_checkpoint})") # <--- MODIFIED
print("="*60)
for key, value in test_results.items():
    if key not in ['epoch', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
# This will also use the best model
print("\nDetailed predictions analysis (from best model):")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\nClassification Report:")
print(classification_report(true_labels, predicted_labels,
                          target_names=[id2label[i] for i in range(num_labels)]))

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


Using device: cuda
Loaded back-translation augmented dataset from disk.


Map:   0%|          | 0/3584 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Dataset ready:
DatasetDict({
    train: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 3584
    })
    test: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 308
    })
})
Labels mapped: {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}
Class weights: tensor([1.1356, 0.5856, 2.4282], device='cuda:0')
Class weights device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Using class weights: tensor([1.1356, 0.5856, 2.4282], device='cuda:0')
Starting training with Focal Loss (evaluating on test set after each epoch)...


W1126 17:00:00.401000 1408 torch/_inductor/utils.py:1558] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted
1,0.537800,0.486794,0.308442,0.357945,0.213163,0.480749,0.519144,0.505783,0.308442
2,0.349000,0.456148,0.652597,0.550295,0.648359,0.544925,0.664662,0.614346,0.652597
3,0.239800,0.528735,0.600649,0.551323,0.611740,0.566065,0.634383,0.550228,0.600649
4,0.138700,0.720281,0.675325,0.584607,0.675261,0.582068,0.675241,0.587313,0.675325
5,0.093400,0.884269,0.672078,0.617073,0.679654,0.603844,0.693925,0.637457,0.672078
6,0.082900,1.366004,0.636364,0.600974,0.648414,0.577820,0.686849,0.655812,0.636364
7,0.031300,1.845639,0.720779,0.637225,0.712222,0.649666,0.709118,0.630513,0.720779
8,0.016600,1.847473,0.688312,0.631633,0.690236,0.633896,0.692548,0.629940,0.688312
9,0.016200,1.744025,0.678571,0.628736,0.683264,0.629459,0.690066,0.630288,0.678571
10,0.002800,1.750035,0.675325,0.623193,0.680178,0.619858,0.687240,0.628670,0.675325


Training completed!



FINAL TEST RESULTS (from best epoch: ModernBERT_QEvasion_model/checkpoint-3136)
eval_loss: 1.8456
eval_accuracy: 0.7208
eval_f1_macro: 0.6372
eval_f1_weighted: 0.7122
eval_precision_macro: 0.6497
eval_precision_weighted: 0.7091
eval_recall_macro: 0.6305
eval_recall_weighted: 0.7208

Detailed predictions analysis (from best model):

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.56      0.44      0.50        79
     Ambivalent       0.78      0.84      0.81       206
Clear Non-Reply       0.61      0.61      0.61        23

       accuracy                           0.72       308
      macro avg       0.65      0.63      0.64       308
   weighted avg       0.71      0.72      0.71       308


Confusion Matrix:
[[ 35  41   3]
 [ 27 173   6]
 [  0   9  14]]


In [4]:
# Access the specific metric (HuggingFace adds 'eval_' prefix to metrics in the results dict)
final_f1_score = test_results.get('eval_f1_macro', 0)
target_threshold = 0.70

print(f"Achieved F1 Macro: {final_f1_score:.4f}")
print(f"Target Threshold:  {target_threshold}")

if final_f1_score >= target_threshold:
    print("\nSUCCESS: Threshold met. Saving model to Google Drive...")

    # Create the directory if it doesn't exist
    if not os.path.exists(drive_save_path):
        os.makedirs(drive_save_path)

    # Save the model and tokenizer using the Trainer's save_model method
    # This ensures both config and weights of the *best* loaded model are saved
    trainer.save_model(drive_save_path)
    tokenizer.save_pretrained(drive_save_path)

    print(f"Model successfully saved to: {drive_save_path}")

else:
    print(f"\nSKIP: Score {final_f1_score:.4f} did not meet the requirement of {target_threshold}.")
    print("Model was NOT saved to Drive.")

Achieved F1 Macro: 0.6372
Target Threshold:  0.7

SKIP: Score 0.6372 did not meet the requirement of 0.7.
Model was NOT saved to Drive.


In [5]:
from google.colab import runtime

print("Nigga money")
runtime.unassign()

Nigga money
